# 06 - Explainability & Interpretability

This notebook provides model interpretation for publication:
1. Load final model and selected features
2. Compute feature importance
3. Generate biomarker ranking
4. Run SHAP analysis (Summary, Waterfall, Dependence plots)
5. Save all figures and tables

> **Note:** All logic is in `src/explainability.py`. This notebook only orchestrates.

In [28]:
import sys
from pathlib import Path

project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

import pandas as pd
import matplotlib.pyplot as plt

import config
from src.io import load_model, save_figure, save_table, logger
from src.models import xgb_safe_frame, xgb_feature_name_map
from src.explainability import (
    compute_feature_importance,
    get_biomarker_ranking,
    run_explainability,
)
from src.visualization import setup_style, plot_feature_importance

setup_style()

## Step 1: Load Final Model and Selected Features

In [29]:
# Load the best model (from Model Training notebook)
model = load_model("best_model_xgboost.joblib")

# Load selected features
selected_df = pd.read_csv(config.TABLES_DIR / "selected_features.csv")
selected_features = selected_df["feature"].tolist()

# Load test data
X_test_selected = pd.read_csv(config.PROCESSED_DIR / "X_test_selected.csv")
y_test = pd.read_csv(config.PROCESSED_DIR / "y_test.csv").iloc[:, 0]

print(f"Model loaded: {type(model).__name__}")
print(f"Selected features: {len(selected_features)}")
print(f"Test samples: {len(X_test_selected)}")

2026-08-06 01:49:55 | INFO     | prostate_bcr | Loading model from D:\Prostate_BCR_Q1\core\outputs\models\best_model_xgboost.joblib


Model loaded: XGBClassifier
Selected features: 30
Test samples: 86


## Step 2: Feature Importance

In [30]:
# Map XGBoost-safe names back to original feature names
name_map = xgb_feature_name_map(selected_features)

importance_df = compute_feature_importance(
    model,
    selected_features,
    top_k=20,
)

print("Top 20 Features by Importance:")
print(importance_df.to_string(index=False))

2026-08-06 01:49:55 | INFO     | prostate_bcr | Extracted feature importance: 30 features


Top 20 Features by Importance:
                                             feature  importance
Primary Lymph Node Presentation Assessment Ind-3_YES    0.125468
                                               PRAG1    0.063646
                                               PRR15    0.050516
                                                MND1    0.049756
                                                RHOC    0.049340
                                                BMP5    0.046670
                                             CCDC127    0.039883
                                                PIM2    0.039650
                                               CXCL8    0.037025
                                               RUSF1    0.033134
                                             FAM107B    0.032368
                                                CHFR    0.032124
                                              CNKSR1    0.032008
                                              KIF1BP    0.0

In [31]:
fig = plot_feature_importance(
    importance_df,
    top_k=20,
    title="Top 20 Features — XGBoost Importance",
    filename="feature_importance_top20.png",
)
plt.show()

2026-08-06 01:49:55 | INFO     | prostate_bcr | Saved figure → D:\Prostate_BCR_Q1\core\outputs\figures\feature_importance_top20.png
2026-08-06 01:49:55 | INFO     | prostate_bcr | Feature importance plot generated (20 features)


## Step 3: Biomarker Ranking

Separate clinical and gene features for biomarker analysis.

In [32]:
from src.preprocessing import identify_column_groups

clinical_cols, gene_cols = identify_column_groups(X_test_selected)

# Rank gene biomarkers
gene_importance = importance_df[importance_df["feature"].isin(gene_cols)].copy()
gene_ranking = get_biomarker_ranking(gene_importance, feature_type="gene")

# Rank clinical biomarkers
clinical_importance = importance_df[importance_df["feature"].isin(clinical_cols)].copy()
clinical_ranking = get_biomarker_ranking(clinical_importance, feature_type="clinical")

print("Top Gene Biomarkers:")
print(gene_ranking.head(10).to_string(index=False))

print("\nTop Clinical Biomarkers:")
print(clinical_ranking.head(10).to_string(index=False))

2026-08-06 01:49:55 | INFO     | prostate_bcr | Column groups: 2 clinical, 28 gene
2026-08-06 01:49:55 | INFO     | prostate_bcr | Biomarker ranking: 19 gene features
2026-08-06 01:49:55 | INFO     | prostate_bcr | Biomarker ranking: 1 clinical features


Top Gene Biomarkers:
 rank feature  importance feature_type
    1   PRAG1    0.063646         gene
    2   PRR15    0.050516         gene
    3    MND1    0.049756         gene
    4    RHOC    0.049340         gene
    5    BMP5    0.046670         gene
    6 CCDC127    0.039883         gene
    7    PIM2    0.039650         gene
    8   CXCL8    0.037025         gene
    9   RUSF1    0.033134         gene
   10 FAM107B    0.032368         gene

Top Clinical Biomarkers:
 rank                                              feature  importance feature_type
    1 Primary Lymph Node Presentation Assessment Ind-3_YES    0.125468     clinical


In [33]:
save_table(gene_ranking, "biomarker_ranking_genes.csv", index=False)
save_table(clinical_ranking, "biomarker_ranking_clinical.csv", index=False)
save_table(importance_df, "feature_importance_full.csv", index=False)

print("Biomarker rankings saved to outputs/tables/")

2026-08-06 01:49:55 | INFO     | prostate_bcr | Saved 19 rows → D:\Prostate_BCR_Q1\core\outputs\tables\biomarker_ranking_genes.csv
2026-08-06 01:49:55 | INFO     | prostate_bcr | Saved 1 rows → D:\Prostate_BCR_Q1\core\outputs\tables\biomarker_ranking_clinical.csv
2026-08-06 01:49:55 | INFO     | prostate_bcr | Saved 20 rows → D:\Prostate_BCR_Q1\core\outputs\tables\feature_importance_full.csv


Biomarker rankings saved to outputs/tables/


## Step 4: SHAP Analysis

> **Note:** SHAP analysis requires the `shap` package.
> If not installed, run: `pip install shap`

In [34]:
# Run full explainability pipeline with SHAP
results = run_explainability(
    model=model,
    X_test=X_test_selected,
    feature_names=selected_features,
    top_k=20,
    run_shap=True,
    sample_index=0,
)

if "shap_error" in results:
    print(f"SHAP analysis skipped: {results['shap_error']}")
else:
    print("SHAP analysis completed successfully.")

2026-08-06 01:49:55 | INFO     | prostate_bcr | Extracted feature importance: 30 features
2026-08-06 01:49:55 | INFO     | prostate_bcr | Biomarker ranking: 20 gene features
2026-08-06 01:49:55 | INFO     | prostate_bcr | Computed SHAP values: 86 samples × 30 features
2026-08-06 01:49:56 | INFO     | prostate_bcr | Created SHAP summary plot
2026-08-06 01:49:56 | INFO     | prostate_bcr | Created SHAP waterfall plot for sample 0
2026-08-06 01:49:56 | INFO     | prostate_bcr | Explainability pipeline complete


SHAP analysis completed successfully.


In [35]:
if "summary_plot" in results:
    fig = results["summary_plot"]
    save_figure(fig, "shap_summary_plot.png")
    plt.show()
else:
    print("SHAP summary plot not available.")

2026-08-06 01:49:56 | INFO     | prostate_bcr | Saved figure → D:\Prostate_BCR_Q1\core\outputs\figures\shap_summary_plot.png


In [36]:
if "waterfall_plot" in results:
    fig = results["waterfall_plot"]
    save_figure(fig, "shap_waterfall_sample0.png")
    plt.show()
else:
    print("SHAP waterfall plot not available.")

2026-08-06 01:49:56 | INFO     | prostate_bcr | Saved figure → D:\Prostate_BCR_Q1\core\outputs\figures\shap_waterfall_sample0.png


## Step 5: SHAP Dependence Plot (Top Feature)

In [37]:
if "shap_values" in results:
    from src.explainability import plot_shap_dependence

    top_feature = importance_df.iloc[0]["feature"]
    fig = plot_shap_dependence(
        results["shap_values"],
        X_test_selected,
        feature_name=top_feature,
        figsize=(8, 6),
    )
    save_figure(fig, f"shap_dependence_{top_feature[:20]}.png")
    plt.show()
    print(f"Dependence plot generated for: {top_feature}")
else:
    print("SHAP values not available.")

2026-08-06 01:49:56 | INFO     | prostate_bcr | Created SHAP dependence plot for feature: Primary Lymph Node Presentation Assessment Ind-3_YES
2026-08-06 01:49:57 | INFO     | prostate_bcr | Saved figure → D:\Prostate_BCR_Q1\core\outputs\figures\shap_dependence_Primary Lymph Node P.png


<Figure size 2400x1800 with 0 Axes>

Dependence plot generated for: Primary Lymph Node Presentation Assessment Ind-3_YES


## Summary

| Output | Location |
|--------|----------|
| Feature Importance Table | `outputs/tables/feature_importance_full.csv` |
| Gene Biomarker Ranking | `outputs/tables/biomarker_ranking_genes.csv` |
| Clinical Biomarker Ranking | `outputs/tables/biomarker_ranking_clinical.csv` |
| Feature Importance Plot | `outputs/figures/feature_importance_top20.png` |
| SHAP Summary Plot | `outputs/figures/shap_summary_plot.png` |
| SHAP Waterfall Plot | `outputs/figures/shap_waterfall_sample0.png` |
| SHAP Dependence Plot | `outputs/figures/shap_dependence_*.png` |

